In [32]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted

In [33]:
class MyKnn(BaseEstimator, ClassifierMixin):
    def __init__(self, k=3, dist='euclidean'):
        self.k = k
        self.dist = dist

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.X_train_ = X
        self.y_train_ = y
        return self

    def predict(self, X):
        check_is_fitted(self)
        X = check_array(X)
        return np.array([self._predict(x) for x in X])
    
    def score(self, X, y):
        X, y = check_X_y(X, y)
        y_pred = self.predict(X)
        return np.mean(y_pred == y)
        
    def _dist_type(self, x):
        if self.dist == 'euclidean':
            return np.sqrt(np.sum((self.X_train_ - x)**2, axis=1))
        
        elif self.dist == 'manhattan':
            return np.sum(np.abs(self.X_train_ - x), axis=1)
        
        elif self.dist == 'minkowski':
            p = 3
            return np.sum(np.abs(self.X_train_ - x)**p, axis=1)**(1/p)
        
        elif self.dist == 'chebyshev':
            return np.max(np.abs(self.X_train_ - x), axis=1)

    def _predict(self, x):
        
        nn_idxs = np.argsort(self._dist_type(x))[:self.k]

        labels = self.y_train_[nn_idxs]
        most_occuring_value = max(list(labels), key=list(labels).count)
        return most_occuring_value

In [34]:
from sklearn.datasets import load_iris
import pandas as pd

data = load_iris(as_frame=True).frame

from sklearn.model_selection import train_test_split

X = data.drop('target', axis=1)
y = data[['target']]

X_train, X_test, y_train, y_test = train_test_split(X.values, y.values, test_size = 0.2, random_state=23)

In [35]:
knn = MyKnn(3)
knn.fit(X_train, y_train.ravel())
print(knn.predict(np.array([[5.1, 3.5, 1.4, 0.2]])))

[0]


In [36]:
euc = MyKnn(k=3, dist='euclidean').fit(X_train, y_train.ravel())
mht = MyKnn(k=3, dist='manhattan').fit(X_train, y_train.ravel())
mkv = MyKnn(k=3, dist='minkowski').fit(X_train, y_train.ravel())
cby = MyKnn(k=3, dist='chebyshev').fit(X_train, y_train.ravel())

print(euc.score(X_test, y_test.ravel()))
print(mht.score(X_test, y_test.ravel()))
print(mkv.score(X_test, y_test.ravel()))
print(cby.score(X_test, y_test.ravel()))

0.9666666666666667
0.9666666666666667
0.9666666666666667
0.9666666666666667
